In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_7.py ──
"""
Shared infrastructure for MLFP04 Exercise 7 — Recommender Systems.

Scenario: Singapore e-commerce marketplace (similar to Shopee/Lazada).
  - 100 users x 50 SKUs from a fictional SG electronics retailer
  - 30% ratings observed (explicit 1-5 stars)
  - Holdout 20% of observed ratings for evaluation
  - Item features = 8 latent product attributes (price tier, category, etc.)

Contains: synthetic rating matrix generation, evaluation metrics (RMSE,
precision@k, MAP), shared constants, and output-dir setup.

Technique-specific code (content-based profile, user/item CF, ALS, hybrid
blending) lives in the per-technique files.
"""

from pathlib import Path

import numpy as np
import polars as pl


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex7_recommenders"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# CONSTANTS — synthetic SG e-commerce dataset
# ════════════════════════════════════════════════════════════════════════

N_USERS = 100
N_ITEMS = 50
N_LATENT_TRUE = 5
N_ITEM_FEATURES = 8
SPARSITY = 0.30
HOLDOUT_FRAC = 0.20
RNG_SEED = 42

RATING_MIN = 1.0
RATING_MAX = 5.0
RELEVANCE_THRESHOLD = 3.5  # ratings >= 3.5 are "relevant" for ranking metrics


# ════════════════════════════════════════════════════════════════════════
# SYNTHETIC RATING MATRIX — shared across every technique
# ════════════════════════════════════════════════════════════════════════


def build_rating_dataset(
    seed: int = RNG_SEED,
) -> dict:
    """Generate the shared synthetic e-commerce rating matrix.

    Returns a dict with:
      R_observed       — (N_USERS, N_ITEMS) with NaN where not rated
      R_train          — R_observed with holdout entries set to NaN
      mask             — boolean observed mask
      train_mask       — boolean training-only mask
      holdout_mask     — boolean holdout mask
      U_true, V_true   — ground-truth latent factors (for subspace recovery)
      item_features    — (N_ITEMS, N_ITEM_FEATURES) content-feature matrix
      user_ids, item_ids — stable string IDs
      ratings_df       — long-format polars DataFrame of observed ratings
    """
    rng = np.random.default_rng(seed=seed)

    U_true = rng.normal(0, 1, size=(N_USERS, N_LATENT_TRUE))
    V_true = rng.normal(0, 1, size=(N_ITEMS, N_LATENT_TRUE))

    R_full = U_true @ V_true.T
    R_full = (R_full - R_full.min()) / (R_full.max() - R_full.min()) * 4 + 1
    R_full += rng.normal(0, 0.3, size=R_full.shape)
    R_full = np.clip(R_full, RATING_MIN, RATING_MAX)

    mask = rng.random(size=(N_USERS, N_ITEMS)) < SPARSITY
    R_observed = np.where(mask, R_full, np.nan)

    holdout_mask = mask & (rng.random(size=(N_USERS, N_ITEMS)) < HOLDOUT_FRAC)
    train_mask = mask & ~holdout_mask
    R_train = np.where(train_mask, R_observed, np.nan)

    item_features = rng.random(size=(N_ITEMS, N_ITEM_FEATURES))

    user_ids = [f"sg_user_{i:03d}" for i in range(N_USERS)]
    item_ids = [f"sku_{j:02d}" for j in range(N_ITEMS)]

    rows = []
    for i in range(N_USERS):
        for j in range(N_ITEMS):
            if mask[i, j]:
                rows.append(
                    {
                        "user_id": user_ids[i],
                        "item_id": item_ids[j],
                        "rating": round(float(R_observed[i, j]), 1),
                        "in_holdout": bool(holdout_mask[i, j]),
                    }
                )
    ratings_df = pl.DataFrame(rows)

    return {
        "R_observed": R_observed,
        "R_train": R_train,
        "mask": mask,
        "train_mask": train_mask,
        "holdout_mask": holdout_mask,
        "U_true": U_true,
        "V_true": V_true,
        "item_features": item_features,
        "user_ids": user_ids,
        "item_ids": item_ids,
        "ratings_df": ratings_df,
        "rng": rng,
    }


# ════════════════════════════════════════════════════════════════════════
# EVALUATION METRICS — shared across every technique
# ════════════════════════════════════════════════════════════════════════


def holdout_rmse(
    predictions: np.ndarray,
    R_true: np.ndarray,
    holdout_mask: np.ndarray,
) -> tuple[float, float]:
    """Return (RMSE, coverage) on the holdout mask.

    Coverage = fraction of holdout pairs where the method produced a
    non-NaN prediction. Methods that can't predict cold pairs have
    coverage < 1.0.
    """
    errors = []
    n_holdout = int(holdout_mask.sum())
    for i in range(predictions.shape[0]):
        for j in range(predictions.shape[1]):
            if holdout_mask[i, j] and not np.isnan(predictions[i, j]):
                errors.append((R_true[i, j] - predictions[i, j]) ** 2)
    if not errors:
        return float("inf"), 0.0
    rmse = float(np.sqrt(np.mean(errors)))
    coverage = len(errors) / max(n_holdout, 1)
    return rmse, coverage


def precision_at_k(
    predictions: np.ndarray,
    R_true: np.ndarray,
    holdout_mask: np.ndarray,
    k: int = 5,
    threshold: float = RELEVANCE_THRESHOLD,
) -> float:
    """Precision@k averaged across users.

    For each user, rank holdout items by predicted score, take the top-k,
    and compute what fraction of those have true rating >= threshold.
    """
    precisions = []
    for u in range(predictions.shape[0]):
        holdout_items = np.where(holdout_mask[u])[0]
        if len(holdout_items) == 0:
            continue
        relevant = {j for j in holdout_items if R_true[u, j] >= threshold}
        if not relevant:
            continue
        scored = [
            (j, predictions[u, j])
            for j in holdout_items
            if not np.isnan(predictions[u, j])
        ]
        scored.sort(key=lambda x: -x[1])
        top = [j for j, _ in scored[:k]]
        if not top:
            continue
        precisions.append(len(set(top) & relevant) / len(top))
    return float(np.mean(precisions)) if precisions else 0.0


def mean_average_precision(
    predictions: np.ndarray,
    R_true: np.ndarray,
    holdout_mask: np.ndarray,
    threshold: float = RELEVANCE_THRESHOLD,
) -> float:
    """Mean Average Precision across users on the holdout set."""
    aps = []
    for u in range(predictions.shape[0]):
        holdout_items = np.where(holdout_mask[u])[0]
        if len(holdout_items) == 0:
            continue
        relevant = {j for j in holdout_items if R_true[u, j] >= threshold}
        if not relevant:
            continue
        scored = [
            (j, predictions[u, j])
            for j in holdout_items
            if not np.isnan(predictions[u, j])
        ]
        scored.sort(key=lambda x: -x[1])

        hits = 0
        sum_precision = 0.0
        for rank, (j, _) in enumerate(scored, 1):
            if j in relevant:
                hits += 1
                sum_precision += hits / rank
        if hits > 0:
            aps.append(sum_precision / len(relevant))
    return float(np.mean(aps)) if aps else 0.0


def print_method_scores(
    name: str, preds: np.ndarray, R_true: np.ndarray, holdout_mask: np.ndarray
) -> dict:
    """Print a single-line scorecard and return the metrics dict."""
    rmse, cov = holdout_rmse(preds, R_true, holdout_mask)
    p5 = precision_at_k(preds, R_true, holdout_mask, k=5)
    m = mean_average_precision(preds, R_true, holdout_mask)
    print(
        f"  {name:<22} RMSE={rmse:6.4f}  coverage={cov:6.1%}  "
        f"P@5={p5:6.4f}  MAP={m:6.4f}"
    )
    return {"RMSE": rmse, "Coverage": cov, "P@5": p5, "MAP": m}


# ════════════════════════════════════════════════════════════════════════
# VISUAL HELPERS
# ════════════════════════════════════════════════════════════════════════


def save_html(fig, filename: str) -> Path:
    """Write a plotly fig into OUTPUT_DIR and return the path."""
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    print(f"  saved: {path}")
    return path




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 7.1: Content-Based Filtering
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Build a user profile from the items they already rated
#   - Score unseen items by cosine similarity to that profile
#   - Understand why content-based filtering handles cold-start items
#   - See where it FAILS: cold-start users, narrow interest tunnels
#
# PREREQUISITES: MLFP04 Exercise 3 (cosine similarity, vector embeddings)
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — item features + user profile intuition
#   2. Build — content-based prediction function
#   3. Train — no training loop; profiles are computed at inference
#   4. Visualise — predicted vs observed ratings scatter
#   5. Apply — Singapore e-commerce new-SKU launch
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import polars as pl
from kailash_ml import ModelVisualizer



## THEORY — Why content-based works


In [ ]:
# Every item has a feature vector. Every user has a rating history. The
# user's "taste profile" is a weighted sum of the features of items they
# rated, with weights = ratings. To score a new item, cosine-similarity
# it against the profile.
#
# STRENGTHS: works for cold-start items, explainable, no cross-user leakage
# WEAKNESSES: cold-start users (no profile), filter bubble, feature quality



## TASK 2 — BUILD the content-based predictor


Predict ratings via cosine similarity to the user's taste profile.


In [ ]:
def content_based_predict(
    R: np.ndarray,
    item_feats: np.ndarray,
    obs_mask: np.ndarray,
) -> np.ndarray:
    n_users, n_items = R.shape
    predictions = np.full((n_users, n_items), np.nan)

    for u in range(n_users):
        rated_idx = np.where(obs_mask[u])[0]
        if len(rated_idx) == 0:
            continue

        # TODO: Build the user's taste profile as the rating-weighted sum of
        # item feature vectors for items they rated. Hint:
        #   ratings_u = np.nan_to_num(R[u, rated_idx], nan=0.0)
        #   profile = (ratings_u[:, None] * item_feats[rated_idx]).sum(axis=0)
        profile = ____

        # TODO: Normalise the profile to unit length (guard against zero norm)
        profile_norm = np.linalg.norm(profile)
        if profile_norm < 1e-10:
            continue
        profile = ____

        for j in range(n_items):
            feat_norm = np.linalg.norm(item_feats[j])
            if feat_norm < 1e-10:
                continue
            # TODO: cosine similarity between profile and item_feats[j]
            sim = ____
            # Map cosine [-1, 1] to rating [1, 5]
            predictions[u, j] = 1.0 + (sim + 1.0) * 2.0

    return predictions



## TASK 3 — "TRAIN" (content-based has no training loop)


In [ ]:
print("\n" + "=" * 70)
print("  Content-Based Filtering on SG E-commerce Ratings")
print("=" * 70)

data = build_rating_dataset()
R_train = data["R_train"]
R_observed = data["R_observed"]
train_mask = data["train_mask"]
holdout_mask = data["holdout_mask"]
item_features = data["item_features"]

# TODO: Call content_based_predict on R_train with item_features and train_mask
cb_predictions = ____



### Checkpoint


In [ ]:
cb_rmse, cb_coverage = holdout_rmse(cb_predictions, R_observed, holdout_mask)
assert cb_predictions.shape == (N_USERS, N_ITEMS), "Prediction matrix shape"
assert cb_rmse > 0, "Content-based RMSE should be positive"
print(
    "\n[ok] Checkpoint passed — content-based predictor produced holdout "
    f"RMSE={cb_rmse:.4f}, coverage={cb_coverage:.1%}\n"
)



## TASK 4 — VISUALISE predicted vs observed


In [ ]:
viz = ModelVisualizer()

pairs = []
for i in range(N_USERS):
    for j in range(N_ITEMS):
        if holdout_mask[i, j] and not np.isnan(cb_predictions[i, j]):
            pairs.append(
                {
                    "user": f"u{i:03d}",
                    "true": float(R_observed[i, j]),
                    "pred": float(cb_predictions[i, j]),
                }
            )

pair_df = pl.DataFrame(pairs)
# TODO: Build a scatter plot with viz.scatter — x="true", y="pred", color="user"
fig = ____
fig.update_layout(
    title="Content-Based: Predicted vs Observed Rating (holdout)",
    xaxis_title="Observed rating (1-5)",
    yaxis_title="Predicted rating (1-5)",
)
save_html(fig, "01_content_based_scatter.html")

print_method_scores("Content-Based", cb_predictions, R_observed, holdout_mask)



## TASK 5 — APPLY: Singapore E-commerce New-SKU Launch


In [ ]:
# SCENARIO: A Singapore consumer-electronics retailer launches 200 new
# SKUs per week. New SKUs have zero ratings — pure cold-start. CF methods
# all need co-rating history that doesn't exist yet.
#
# Content-based filtering is the ONLY method here that can recommend a
# SKU on day 0 as long as the SKU has a feature vector (price tier,
# category, brand, screen size, etc).
#
# BUSINESS IMPACT: New SKUs earn ~80% of lifetime revenue in the first
# 30 days. Content-based filtering recovers ~60% of the cold-start gap —
# roughly S$72K/month in recovered cross-sell revenue on a 200-SKU/week
# launch cadence.



## REFLECTION


[x] Built a user taste profile from rated items + their ratings
  [x] Scored unseen items via cosine similarity to the profile
  [x] Measured holdout RMSE and ranking metrics (P@5, MAP)
  [x] Understood WHEN to use content-based: cold-start items
  [x] Understood WHEN it fails: cold-start users, narrow graphs

  Next: 02_user_cf.py — community opinions escape the filter bubble.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

